# DistilBERT を SST-2 で fine-tuning して推論する

このノートブックでは、Hugging Face Datasets から直接 import できる公開データ SST-2 を使って、`distilbert/distilbert-base-uncased` を2値分類用に fine-tuning する。

SST-2 は短文の感情分類データで、ラベルは `0 = negative`、`1 = positive`。  
DistilBERT は DeBERTa V3 より小さく、fine-tuning の流れを示すデモでは精度の改善が見えやすい。

In [ ]:
# 必要なら最初に実行。
# %pip install -U transformers datasets accelerate scikit-learn torch

In [ ]:
import numpy as np
import torch
from datasets import load_dataset
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
)

MODEL_NAME = "distilbert/distilbert-base-uncased"
DATASET_NAME = "nyu-mll/glue"
DATASET_CONFIG = "sst2"
LABEL_NAMES = ["negative", "positive"]
id2label = {i: label for i, label in enumerate(LABEL_NAMES)}
label2id = {label: i for i, label in id2label.items()}

## 1. 公開データを読み込む

`load_dataset("nyu-mll/glue", "sst2")` で公開データを直接読み込む。  
初回実行時はデータセットがダウンロードされる。

In [ ]:
raw_dataset = load_dataset(DATASET_NAME, DATASET_CONFIG)
raw_dataset

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/35.3k [00:00<?, ?B/s]

sst2/train-00000-of-00001.parquet:   0%|          | 0.00/3.11M [00:00<?, ?B/s]

sst2/validation-00000-of-00001.parquet:   0%|          | 0.00/72.8k [00:00<?, ?B/s]

sst2/test-00000-of-00001.parquet:   0%|          | 0.00/148k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/67349 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/872 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1821 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['sentence', 'label', 'idx'],
        num_rows: 67349
    })
    validation: Dataset({
        features: ['sentence', 'label', 'idx'],
        num_rows: 872
    })
    test: Dataset({
        features: ['sentence', 'label', 'idx'],
        num_rows: 1821
    })
})

## 2. 実行しやすいサイズに絞る

ここでは学習用に30000件、評価用に validation 全体を使う。  
SST-2 には `train`、`validation`、`test` があるが、`test` はラベルが公開されていないため、ここでは `validation` を評価用に使う。  

In [ ]:
train_dataset = raw_dataset["train"].shuffle(seed=42).select(range(30000))
eval_dataset = raw_dataset["validation"]

print("train label counts:", np.bincount(train_dataset["label"]))
print("eval label counts:", np.bincount(eval_dataset["label"]))

print(train_dataset[0])
print(eval_dataset[0])

train label counts: [13414 16586]
eval label counts: [428 444]
{'sentence': 'klein , charming in comedies like american pie and dead-on in election , ', 'label': 1, 'idx': 32326}
{'sentence': "it 's a charming and often affecting journey . ", 'label': 1, 'idx': 0}


## 3. Tokenizer で前処理する

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(batch):
    return tokenizer(
        batch["sentence"],
        truncation=True,
        max_length=128,
    )

tokenized_train = train_dataset.map(tokenize, batched=True)
tokenized_eval = eval_dataset.map(tokenize, batched=True)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

tokenized_train

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Map:   0%|          | 0/30000 [00:00<?, ? examples/s]

Map:   0%|          | 0/872 [00:00<?, ? examples/s]

Dataset({
    features: ['sentence', 'label', 'idx', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 30000
})

## 4. 分類モデルを作る

`AutoModelForSequenceClassification` を使うと、DistilBERT 本体の上に分類ヘッドが追加さる。  
この分類ヘッドと DistilBERT 本体を、公開データで fine-tuning

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(LABEL_NAMES),
    id2label=id2label,
    label2id=label2id,
)

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert/distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


## 5. fine-tuning する

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels,
        preds,
        average="binary",
        zero_division=0,
    )
    acc = accuracy_score(labels, preds)
    return {
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1,
    }

training_args = TrainingArguments(
    output_dir="./distilbert_sst2_out",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_steps=50,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    greater_is_better=True,
    max_grad_norm=1.0,
    logging_nan_inf_filter=False,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_eval,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.239847,0.275196,0.901376,0.903153,0.903153,0.903153
2,0.184351,0.351769,0.896789,0.883117,0.918919,0.900662
3,0.079158,0.459717,0.894495,0.880952,0.916667,0.898455


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=5625, training_loss=0.16489246554904513, metrics={'train_runtime': 374.8248, 'train_samples_per_second': 240.112, 'train_steps_per_second': 15.007, 'total_flos': 823487723351232.0, 'train_loss': 0.16489246554904513, 'epoch': 3.0})

## 6. 評価する

In [ ]:
trainer.evaluate()

Training Loss,Validation Loss,Epoch,Accuracy,Precision,Recall,F1
0.079158,0.275196,3,0.901376,0.903153,0.903153,0.903153


{'eval_loss': 0.2751956582069397,
 'eval_accuracy': 0.9013761467889908,
 'eval_precision': 0.9031531531531531,
 'eval_recall': 0.9031531531531531,
 'eval_f1': 0.9031531531531531}

## 7. 新しい文章を推論する

In [ ]:
def predict(texts):
    model.eval()
    inputs = tokenizer(
        texts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=128,
    )
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.no_grad():
        logits = model(**inputs).logits
        probs = torch.softmax(logits, dim=-1)
        pred_ids = torch.argmax(probs, dim=-1).cpu().tolist()

    rows = []
    for text, pred_id, prob in zip(texts, pred_ids, probs.cpu().tolist()):
        rows.append({
            "text": text,
            "pred_label": id2label[pred_id],
            "confidence": prob[pred_id],
        })
    return rows

new_texts = [
    "A smart and moving film with excellent performances.",
    "The plot is dull, predictable, and painfully slow.",
    "A charming and funny story.",
    "This was a waste of time.",
]

predict(new_texts)

[{'text': 'A smart and moving film with excellent performances.',
  'pred_label': 'positive',
  'confidence': 0.99834144115448},
 {'text': 'The plot is dull, predictable, and painfully slow.',
  'pred_label': 'negative',
  'confidence': 0.9918299317359924},
 {'text': 'A charming and funny story.',
  'pred_label': 'positive',
  'confidence': 0.9984536170959473},
 {'text': 'This was a waste of time.',
  'pred_label': 'negative',
  'confidence': 0.9890050292015076}]